# TP6 : fine-tuning de ResNet50 et comparaison MLP / CNN / ResNet

Objectif : améliorer la classification des images CIFAR-10 avec un ResNet50 pré-entraîné sur ImageNet. Phase 1 : la tête seule, backbone gelé. Phase 2 : dégel des deux derniers blocs résiduels avec un learning rate 10 fois plus faible sur le backbone. Puis comparaison des trois modèles du parcours sur le même jeu de test (accuracy et F1 par classe) et sauvegarde du meilleur avec ses métadonnées.

L'énoncé complet, avec les critères de réussite, est dans `enonce.md`.

*Complétez les cellules marquées `# TODO`, dans l'ordre.*

## Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [ ]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
torchviz==0.0.3
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

## Préparation (fournie)

**Données** : CIFAR-10 (Krizhevsky, Nair et Hinton, 2009), fourni par torchvision : des photos couleur de 32 x 32 pixels, en 10 classes (avion, automobile, oiseau, chat, cerf, chien, grenouille, cheval, bateau, camion). Le site du jeu ne donne pas de licence ; il demande de citer le rapport *Learning Multiple Layers of Features from Tiny Images* (Krizhevsky, 2009). Tous les TP utilisent les mêmes images : 5000 d'entraînement et 1000 de validation, tirées de l'entraînement de CIFAR-10, et 1000 de test.

Le ResNet a été entraîné sur des images de 224 x 224 : son prétraitement agrandit les images de 32 x 32 à cette taille.

In [ ]:
def preparer_archive_cifar10(racine="data"):
    """Place les fichiers de CIFAR-10 dans racine/cifar-10-batches-py sans passer par le serveur d'origine, lent.

    Dans l'ordre : fichiers déjà décompressés ; archive déposée dans la session (cifar-10-python.tar.gz ou
    cifar-10-batches-py.tar.gz, dans le dossier du notebook ou dans racine) ; archive d'origine copiée sur le
    Google Drive de la formation. Si rien n'aboutit, torchvision télécharge ensuite depuis le serveur d'origine ;
    dans tous les cas, il vérifie l'intégrité des fichiers.
    """
    import tarfile
    import urllib.request
    from pathlib import Path

    if Path(racine, "cifar-10-batches-py").exists():
        return
    noms = ("cifar-10-python.tar.gz", "cifar-10-batches-py.tar.gz")
    deposees = [Path(dossier, nom) for dossier in (".", racine) for nom in noms if Path(dossier, nom).exists()]
    if deposees:
        archive = deposees[0]
        print(f"Archive trouvée : {archive}")
    else:
        archive = Path(racine, "cifar-10-python.tar.gz")
        archive.parent.mkdir(parents=True, exist_ok=True)
        url = ("https://drive.usercontent.google.com/download?id=1kyq-1VhOPNr2Bo0w5DiLYuNghOdOWl0a"
               "&export=download&confirm=t")
        print("Téléchargement de CIFAR-10 depuis le Google Drive de la formation (170 Mo)...")
        try:
            urllib.request.urlretrieve(url, archive)
        except OSError as erreur:
            print(f"Google Drive indisponible ({erreur}) : téléchargement depuis le serveur d'origine.")
            return
    try:
        with tarfile.open(archive) as contenu:
            contenu.extractall(racine, filter="data")
    except tarfile.TarError:
        print(f"{archive} n'est pas une archive valide : téléchargement depuis le serveur d'origine.")


def charger_cifar10(dossier="cifar10", n_train=5000, n_val=1000, n_test=1000, seed=0):
    """Images CIFAR-10 au format ImageFolder : dossier/<partie>/<classe>/<image>.png, classes en français.

    Tirages à graine fixe, les mêmes pour tous les TP : n_train et n_val images prises dans les 50 000 images
    d'entraînement de CIFAR-10, sans recouvrement, et n_test dans les 10 000 images de test. Une partie déjà
    présente n'est pas réécrite ; l'écriture passe par un dossier temporaire propre à chaque exécution, renommé à
    la fin : une écriture interrompue n'est jamais prise pour une partie complète, et deux notebooks lancés en même
    temps sur la même machine ne se gênent pas. Les fichiers de CIFAR-10 sont d'abord
    récupérés par preparer_archive_cifar10 (archive déposée dans la session, ou Google Drive de la formation).
    """
    import shutil
    import tempfile
    from pathlib import Path

    import torch
    from torchvision.datasets import CIFAR10

    noms = ["avion", "automobile", "oiseau", "chat", "cerf", "chien", "grenouille", "cheval", "bateau", "camion"]
    generateur = torch.Generator().manual_seed(seed)
    ordre = torch.randperm(50000, generator=generateur).tolist()
    parties = {"train": (True, ordre[:n_train]),
               "val": (True, ordre[n_train:n_train + n_val]),
               "test": (False, torch.randperm(10000, generator=generateur)[:n_test].tolist())}
    preparer_archive_cifar10()
    for partie, (entrainement, indices) in parties.items():
        if Path(dossier, partie).exists():
            continue
        source = CIFAR10("data", train=entrainement, download=True)
        Path(dossier).mkdir(parents=True, exist_ok=True)
        temporaire = Path(tempfile.mkdtemp(prefix=f"{partie}.tmp", dir=dossier))
        for numero, indice in enumerate(indices):
            image, label = source[indice]
            sortie = Path(temporaire, noms[label], f"{numero:05d}.png")
            sortie.parent.mkdir(parents=True, exist_ok=True)
            image.save(sortie)
        try:
            temporaire.rename(Path(dossier, partie))
        except OSError:
            # Un autre notebook a fini d'écrire cette partie pendant ce temps : on garde la sienne.
            if not Path(dossier, partie).exists():
                raise
            shutil.rmtree(temporaire)

In [ ]:
charger_cifar10()

## Étape 0 : les données et les fonctions (fournies)

In [ ]:
from pathlib import Path

for partie in ("train", "val", "test"):
    effectifs = {c.name: len(list(c.iterdir())) for c in sorted(Path(f"cifar10/{partie}").iterdir())}
    print(partie, sum(effectifs.values()), effectifs)

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.models import ResNet50_Weights, resnet50
from torchvision.transforms import v2

weights = ResNet50_Weights.DEFAULT
preprocess = weights.transforms()
torch.manual_seed(0)
train_loader = DataLoader(ImageFolder("cifar10/train", transform=preprocess), batch_size=32, shuffle=True,
                          num_workers=2)
val_loader = DataLoader(ImageFolder("cifar10/val", transform=preprocess), batch_size=64, num_workers=2)
test_loader = DataLoader(ImageFolder("cifar10/test", transform=preprocess), batch_size=64, num_workers=2)
CLASSES = test_loader.dataset.classes
print(preprocess)

In [ ]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    for x, y in loader:
        loss = loss_fn(model(x.to(device)), y.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            total_loss += loss_fn(logits, y).item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

In [ ]:
from torch.utils.tensorboard import SummaryWriter


def fit(model, optimizer, n_epochs, nom):
    loss_fn = nn.CrossEntropyLoss()
    writer = SummaryWriter(f"runs/tp6/{nom}")
    for epoch in range(1, n_epochs + 1):
        train_one_epoch(model, train_loader, loss_fn, optimizer)
        train_loss, train_acc = evaluate(model, train_loader, loss_fn)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, epoch)
        writer.add_scalars("accuracy", {"train": train_acc, "val": val_acc}, epoch)
        print(f"époque {epoch}  loss {train_loss:.3f}  acc {train_acc:.3f}  val_loss {val_loss:.3f}  val_acc {val_acc:.3f}")
    writer.close()

## Étape 1 : la tête seule, backbone gelé

Chargez `resnet50` avec `weights`, gelez tous ses paramètres et remplacez `fc` par une couche à 10 sorties.

In [ ]:
torch.manual_seed(0)
resnet = resnet50(weights=weights, progress=False)
# TODO : geler tous les paramètres : p.requires_grad_(False) pour chaque p de resnet.parameters()
...
# TODO : remplacer resnet.fc par nn.Linear(resnet.fc.in_features, 10)
...
resnet = resnet.to(device)
assert sum(p.numel() for p in resnet.parameters() if p.requires_grad) == 2048 * 10 + 10

Entraînez 5 époques avec Adam (lr=1e-3) sur les seuls paramètres de `fc`.

In [ ]:
# TODO : fit(resnet, Adam sur resnet.fc.parameters() avec lr=1e-3, n_epochs=5, nom="tete-seule")

## Étape 2 : dégeler les deux derniers blocs

Dégelez `layer3` et `layer4`.

In [ ]:
# TODO : requires_grad_(True) sur les paramètres de layer3 et layer4
print(sum(p.numel() for p in resnet.parameters() if p.requires_grad), "paramètres entraînables")

L'optimiseur reçoit deux groupes : `layer3` et `layer4` avec lr=1e-4, dix fois plus faible que la tête (`fc`, lr=1e-3). Entraînez 5 époques.

In [ ]:
optimizer = torch.optim.Adam([
    {"params": list(resnet.layer3.parameters()) + list(resnet.layer4.parameters()), "lr": ...},  # TODO : 1e-4
    {"params": resnet.fc.parameters(), "lr": ...},  # TODO : 1e-3
])
torch.manual_seed(0)
fit(resnet, optimizer, n_epochs=5, nom="degel")

Les deux phases dans TensorBoard (fourni) : `fit` a écrit leurs courbes sous `runs/tp6/<nom>`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/tp6

## Étape 3 : accuracy et F1 par classe

`metriques(model, loader)` remplit la matrice de confusion (ligne = vraie classe, colonne = classe prédite). Complétez le calcul de la precision, du recall et du F1 de chaque classe : precision = vrais positifs / colonne, recall = vrais positifs / ligne.

In [ ]:
def metriques(model, loader):
    model.eval()
    confusion = torch.zeros(10, 10)
    with torch.no_grad():
        for x, y in loader:
            for vraie, predite in zip(y, model(x.to(device)).argmax(1).cpu()):
                confusion[vraie, predite] += 1
    vrais_positifs = confusion.diag()
    # TODO : precision = vrais_positifs / somme des colonnes (confusion.sum(dim=0))
    # TODO : recall = vrais_positifs / somme des lignes (confusion.sum(dim=1))
    # TODO : f1 = 2 * precision * recall / (precision + recall)
    ...
    return round((vrais_positifs.sum() / confusion.sum()).item(), 3), [round(v, 3) for v in f1.tolist()]

In [ ]:
acc, f1 = metriques(resnet, val_loader)
print("ResNet50, validation : accuracy", acc)
print("F1 par classe :", dict(zip(CLASSES, f1)))
assert len(f1) == 10 and all(0 <= v <= 1 for v in f1)

## Étape 4 : comparer MLP, CNN et ResNet sur le même jeu de test (fourni)

Les cellules réentraînent le MLP du TP2 et le CNN du TP3, sur les images en 32 x 32, puis évaluent les trois modèles une seule fois sur les mêmes 1000 images de test.

In [ ]:
from torch.utils.data import TensorDataset

vers_vecteur = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])


def charger_tenseurs(partie):
    dataset = ImageFolder(f"cifar10/{partie}", transform=vers_vecteur)
    return TensorDataset(torch.stack([image for image, _ in dataset]).flatten(1), torch.tensor(dataset.targets))

In [ ]:
torch.manual_seed(0)
mlp_train = DataLoader(charger_tenseurs("train"), batch_size=64, shuffle=True)
mlp = nn.Sequential(nn.Linear(3072, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)
for epoch in range(20):
    train_one_epoch(mlp, mlp_train, nn.CrossEntropyLoss(), optimizer)

In [ ]:
conversion = [v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean=[0.5] * 3, std=[0.5] * 3)]
cnn_train = DataLoader(ImageFolder("cifar10/train", transform=v2.Compose(
    [v2.RandomHorizontalFlip(), v2.RandomCrop(32, padding=4)] + conversion)), batch_size=32, shuffle=True)
cnn_test = DataLoader(ImageFolder("cifar10/test", transform=v2.Compose(conversion)), batch_size=64)

In [ ]:
class CNNCifar(nn.Module):
    def __init__(self):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.bloc2 = nn.Sequential(nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.bloc3 = nn.Sequential(nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classifieur = nn.Sequential(nn.Flatten(), nn.Linear(64 * 4 * 4, 64), nn.ReLU(), nn.Linear(64, 10))

    def forward(self, x):
        return self.classifieur(self.bloc3(self.bloc2(self.bloc1(x))))

In [ ]:
torch.manual_seed(0)
cnn = CNNCifar().to(device)
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)
for epoch in range(10):
    train_one_epoch(cnn, cnn_train, nn.CrossEntropyLoss(), optimizer)

In [ ]:
resultats = {
    "MLP (TP2)": metriques(mlp, DataLoader(charger_tenseurs("test"), batch_size=256)),
    "CNN (TP3)": metriques(cnn, cnn_test),
    "ResNet50": metriques(resnet, test_loader),
}
for nom, (acc, f1) in resultats.items():
    difficile = CLASSES[f1.index(min(f1))]
    print(f"{nom:10s} accuracy test {acc:.3f}  F1 moyen {sum(f1) / len(f1):.3f}  classe la plus difficile : "
          f"{difficile} (F1 {min(f1):.3f})")

Question : quel modèle retenez-vous ? Quelles classes sont les plus difficiles, et pourquoi le F1 par classe en dit-il plus que l'accuracy globale ?

## Étape 5 : sauvegarder le meilleur modèle

Sauvegardez le `state_dict` du ResNet avec ses métadonnées dans `cifar_best.pt`.

In [ ]:
checkpoint = {
    "model_state": ...,  # TODO : le state_dict du ResNet
    "model_name": "resnet50",
    "classes": CLASSES,
    "epoch": 10,
    "test_accuracy": resultats["ResNet50"][0],
    "test_f1_par_classe": resultats["ResNet50"][1],
}
# TODO : torch.save(checkpoint, "cifar_best.pt")

Rechargez-le dans un ResNet50 neuf et vérifiez qu'il retrouve la même accuracy de test.

In [ ]:
charge = torch.load("cifar_best.pt")
nouveau = resnet50()
nouveau.fc = nn.Linear(2048, 10)
# TODO : nouveau.load_state_dict(...) avec le state_dict rangé dans charge
...
nouveau = nouveau.to(device)
print({cle: valeur for cle, valeur in charge.items() if cle not in ("model_state", "test_f1_par_classe")})
assert metriques(nouveau, test_loader)[0] == resultats["ResNet50"][0]

Dans Colab, le fichier disparaît à la fin de la session : téléchargez-le (panneau Fichiers, à gauche) pour le réutiliser au TP7.

## Bonus

**B1. EfficientNet-B0.** Même feature extraction (tête seule, 5 époques) avec `efficientnet_b0`, dont la dernière couche est `classifier[1]`.

In [ ]:
# TODO : efficientnet_b0 gelé, classifier[1] remplacé, 5 époques